# Lab 1: Introduction to HIP Programming Basics

---

## Abstract

<p style="text-align:justify;">In this lab, you will build and run a HIP application to understand the host-driven GPU execution model on AMD GPUs. You will use HIP runtime APIs to initialize the runtime, manage memory, launch a one-dimensional GPU kernel, synchronize execution, and validate results. By compiling and executing the application on an AMD GPU, you will observe kernel execution behavior and GPU hardware characteristics.</p>

**Note:** This tutorial is designed for AMD GPU architectures and works seamlessly with consumer graphics cards and data center accelerators.

## Objectives

After completing this lab, you will be able to:

- Define and launch a one-dimensional HIP GPU kernel to perform data-parallel computation using thread and block indexing
- Apply core HIP runtime APIs to initialize the runtime, select a device, allocate host and device memory, transfer data, launch kernels, synchronize execution, and clean up resources
- Interpret program output to verify correct GPU execution, observe GPU hardware characteristics, and validate kernel results

## Introduction

<p style="text-align:justify;">The AMD ROCm™ software platform provides a comprehensive open-source ecosystem for developing GPU-accelerated applications on AMD GPUs using the Heterogeneous-computing Interface for Portability (HIP) programming model. HIP enables developers to write portable GPU programs while maintaining an explicit and predictable programming model that clearly defines the roles of the host (CPU) and the device (GPU).</p>

### Host-Driven Execution Model

HIP applications follow a **host-driven execution model** in which:
- **CPU** is responsible for: initializing the runtime, selecting the GPU device, allocating memory, transferring data, configuring kernel launches, enforcing synchronization, and handling errors
- **GPU** focuses exclusively on executing parallel kernels using the single instruction, multiple threads (SIMT) execution paradigm

This lab introduces these core HIP execution concepts through a complete end-to-end workflow using a simple data-parallel vector addition application.

## General Flow

```
┌──────────────────────┐     ┌──────────────────────┐     ┌──────────────────────┐
│   Step 1:            │     │   Step 2:            │     │   Step 3:            │
│   Readying the       │ --> │   Reviewing the      │ --> │   Building and       │
│   Environment &      │     │   HIP Application    │     │   Running the HIP    │
│   Setting Workspace  │     │                      │     │   Application        │
└──────────────────────┘     └──────────────────────┘     └──────────────────────┘
```

---

# Step 1: Readying the Environment and Setting the Workspace

Verify that the HIP development environment is available and create a clean workspace.

## 1.1 Verify the HIP compiler availability

This verifies that the HIP compiler required to build GPU workloads is installed. If available, it will display the installation path of `hipcc`.

In [ ]:
!which hipcc

## 1.2 Display the HIP compiler version

This command prints the installed HIP compiler version, confirming that the HIP toolchain is available for compiling GPU applications.

In [ ]:
!hipcc --version

## 1.3 Create the lab workspace

### 1.3.1 Create the lab directory

This creates an isolated workspace so all lab artifacts are organized and reproducible.

In [ ]:
!mkdir -p hip_intro_lab

### 1.3.2 Change into the lab directory

This sets the working directory so that all files are created and compiled in one location.

In [ ]:
%cd hip_intro_lab

## 1.4 Create the HIP source file

Below is the complete HIP program for vector addition. This program demonstrates the entire HIP workflow from initialization through cleanup.

In [ ]:
%%writefile lab.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <cstdlib>

#define HIP_CHECK(cmd) \
do { \
    hipError_t err = cmd; \
    if (err != hipSuccess) { \
        std::cerr << "HIP error: " \
                  << hipGetErrorString(err) \
                  << " at " << __FILE__ \
                  << ":" << __LINE__ << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

/* ================================
   Device Kernel: Vector Addition
   ================================ */
__global__
void vectorAdd(const float* A,
               const float* B,
               float* C,
               int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        C[idx] = A[idx] + B[idx];
    }
}

int main()
{
    /* ===== Runtime Initialization and Device Setup ===== */
    HIP_CHECK( hipInit(0) );

    int deviceCount;
    HIP_CHECK( hipGetDeviceCount(&deviceCount) );
    if (deviceCount == 0) {
        std::cerr << "No HIP devices available\n";
        return EXIT_FAILURE;
    }

    HIP_CHECK( hipSetDevice(0) );

    hipDeviceProp_t props;
    HIP_CHECK( hipGetDeviceProperties(&props, 0) );

    std::cout << "GPU: " << props.name << "\n";
    std::cout << "Architecture: " << props.gcnArchName << "\n";
    std::cout << "Compute Units: " << props.multiProcessorCount << "\n";
    std::cout << "Global Memory (MB): "
              << props.totalGlobalMem / (1024 * 1024) << "\n\n";

    /* ===== Host and Device Memory ===== */
    const int N = 1024;
    size_t size = N * sizeof(float);

    float* hA = new float[N];
    float* hB = new float[N];
    float* hC = new float[N];

    for (int i = 0; i < N; i++) {
        hA[i] = static_cast<float>(i);
        hB[i] = 2.0f * static_cast<float>(i);
    }

    float *dA, *dB, *dC;
    HIP_CHECK( hipMalloc(&dA, size) );
    HIP_CHECK( hipMalloc(&dB, size) );
    HIP_CHECK( hipMalloc(&dC, size) );
    HIP_CHECK( hipMemset(dC, 0, size) );

    /* ===== Streams and Asynchronous Execution ===== */
    hipStream_t stream;
    HIP_CHECK( hipStreamCreate(&stream) );

    HIP_CHECK( hipMemcpyAsync(dA, hA, size,
                              hipMemcpyHostToDevice, stream) );
    HIP_CHECK( hipMemcpyAsync(dB, hB, size,
                              hipMemcpyHostToDevice, stream) );

    /* ===== Events for Kernel Timing ===== */
    hipEvent_t start, stop;
    HIP_CHECK( hipEventCreate(&start) );
    HIP_CHECK( hipEventCreate(&stop) );

    HIP_CHECK( hipEventRecord(start, stream) );

    dim3 block(256);
    dim3 grid((N + block.x - 1) / block.x);

    hipLaunchKernelGGL(
        vectorAdd, grid, block, 0, stream,
        dA, dB, dC, N
    );

    HIP_CHECK( hipEventRecord(stop, stream) );
    HIP_CHECK( hipEventSynchronize(stop) );

    float elapsedMs = 0.0f;
    HIP_CHECK( hipEventElapsedTime(&elapsedMs, start, stop) );
    std::cout << "Kernel time (ms): " << elapsedMs << "\n";

    /* ===== Result Retrieval and Validation ===== */
    HIP_CHECK( hipMemcpyAsync(hC, dC, size,
                              hipMemcpyDeviceToHost, stream) );
    HIP_CHECK( hipStreamSynchronize(stream) );

    bool pass = true;
    for (int i = 0; i < N; i++) {
        if (hC[i] != hA[i] + hB[i]) {
            pass = false;
            break;
        }
    }

    std::cout << "Vector Add Result: "
              << (pass ? "PASS" : "FAIL") << "\n";

    /* ===== Cleanup ===== */
    HIP_CHECK( hipEventDestroy(start) );
    HIP_CHECK( hipEventDestroy(stop) );
    HIP_CHECK( hipStreamDestroy(stream) );

    HIP_CHECK( hipFree(dA) );
    HIP_CHECK( hipFree(dB) );
    HIP_CHECK( hipFree(dC) );

    delete[] hA;
    delete[] hB;
    delete[] hC;

    HIP_CHECK( hipDeviceReset() );

    return EXIT_SUCCESS;
}

---

# Step 2: Reviewing the HIP Application

Review the HIP application to understand the complete execution flow and host-device coordination. Each section below explains a critical component of the HIP programming model.

## 2.1 Error Handling Macro

```cpp
#define HIP_CHECK(cmd) \
do { \
    hipError_t err = cmd; \
    if (err != hipSuccess) { \
        std::cerr << "HIP error: " \
                  << hipGetErrorString(err) \
                  << " at " << __FILE__ \
                  << ":" << __LINE__ << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
```

<p style="text-align:justify;">The HIP_CHECK macro is a defensive programming pattern used throughout the application to handle GPU runtime errors systematically. Every HIP runtime API call returns a hipError_t status code indicating success or failure. This macro wraps each HIP API call, automatically checks the returned error code, and if the operation failed, it prints a detailed error message including the human-readable error description, the source file name, and the line number where the error occurred. The program then terminates immediately with EXIT_FAILURE to prevent cascading errors. This approach ensures that any GPU operation failure is caught immediately at the point of occurrence, making debugging much easier compared to silent failures that might only manifest later in the execution. The do-while(0) construct is a standard C idiom that allows the macro to be used like a regular function call with a semicolon, maintaining proper syntax even when used in conditional statements.</p>

## 2.2 Device Kernel: Vector Addition

```cpp
__global__
void vectorAdd(const float* A,
               const float* B,
               float* C,
               int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        C[idx] = A[idx] + B[idx];
    }
}
```

<p style="text-align:justify;">This is the GPU kernel function that performs the actual parallel computation on the device. The `__global__` keyword indicates that this function will be executed on the GPU and can be called from host code. The kernel implements a simple element-wise vector addition using a one-thread-per-element mapping strategy. Each thread computes a unique global index by combining its position within a block (threadIdx.x) with the block's position in the grid (blockIdx.x) and the block size (blockDim.x). This indexing formula `idx = blockIdx.x * blockDim.x + threadIdx.x` is fundamental to GPU programming - it allows thousands of threads to work simultaneously on different data elements without conflicts. The bounds check `if (idx < n)` is essential because the total number of threads launched might exceed the actual data size due to block size alignment requirements. Each thread reads one element from arrays A and B, performs the addition, and writes the result to array C - all happening in parallel across hundreds or thousands of GPU threads simultaneously, which is the core of GPU acceleration.</p>

## 2.3 Runtime Initialization and Device Setup

```cpp
HIP_CHECK( hipInit(0) );

int deviceCount;
HIP_CHECK( hipGetDeviceCount(&deviceCount) );
if (deviceCount == 0) {
    std::cerr << "No HIP devices available\n";
    return EXIT_FAILURE;
}

HIP_CHECK( hipSetDevice(0) );

hipDeviceProp_t props;
HIP_CHECK( hipGetDeviceProperties(&props, 0) );

std::cout << "GPU: " << props.name << "\n";
std::cout << "Architecture: " << props.gcnArchName << "\n";
std::cout << "Compute Units: " << props.multiProcessorCount << "\n";
std::cout << "Global Memory (MB): "
          << props.totalGlobalMem / (1024 * 1024) << "\n\n";
```

<p style="text-align:justify;">This section demonstrates the host-driven initialization sequence that every HIP application must perform before executing GPU code. The hipInit call initializes the HIP runtime system and establishes communication between the CPU and GPU. The program then queries how many GPU devices are available in the system using hipGetDeviceCount - this is important for multi-GPU systems where you might want to select a specific device. After verifying that at least one GPU is present, hipSetDevice explicitly selects which GPU to use (device 0 in this case). The hipGetDeviceProperties call retrieves detailed hardware characteristics of the selected GPU, including the device name, architecture identifier, number of compute units (parallel processing cores), and total global memory. This information is then printed to the console, which serves two purposes: it confirms that the GPU was successfully detected and initialized, and it provides valuable information about the hardware capabilities that will execute your code. This host-driven model emphasizes that the CPU maintains complete control over GPU resource management and execution orchestration.</p>

## 2.4 Host and Device Memory Management

```cpp
const int N = 1024;
size_t size = N * sizeof(float);

float* hA = new float[N];
float* hB = new float[N];
float* hC = new float[N];

for (int i = 0; i < N; i++) {
    hA[i] = static_cast<float>(i);
    hB[i] = 2.0f * static_cast<float>(i);
}

float *dA, *dB, *dC;
HIP_CHECK( hipMalloc(&dA, size) );
HIP_CHECK( hipMalloc(&dB, size) );
HIP_CHECK( hipMalloc(&dC, size) );
HIP_CHECK( hipMemset(dC, 0, size) );
```

<p style="text-align:justify;">This section illustrates HIP's explicit memory model, which clearly separates host (CPU) memory from device (GPU) memory. The program first allocates three arrays on the host using standard C++ new operators - these arrays (hA, hB, hC) reside in system RAM and are directly accessible by the CPU. The host arrays are then initialized with test data: array A contains sequential values 0, 1, 2, ..., N-1, while array B contains doubled values 0, 2, 4, ..., 2*(N-1). Next, the program allocates corresponding arrays on the GPU device using hipMalloc, which is analogous to malloc but allocates memory in the GPU's global memory space. These device pointers (dA, dB, dC) cannot be directly dereferenced by CPU code - they are only valid on the GPU. The hipMemset call initializes the device output array dC to zero, ensuring predictable starting conditions. This explicit separation between host and device memory is fundamental to GPU programming - data must be explicitly transferred between these memory spaces using specific HIP API calls, and programmers must carefully manage where data resides and when transfers occur. This design gives developers precise control over data movement, which is critical for performance optimization.</p>

## 2.5 Streams and Asynchronous Execution

```cpp
hipStream_t stream;
HIP_CHECK( hipStreamCreate(&stream) );

HIP_CHECK( hipMemcpyAsync(dA, hA, size,
                          hipMemcpyHostToDevice, stream) );
HIP_CHECK( hipMemcpyAsync(dB, hB, size,
                          hipMemcpyHostToDevice, stream) );
```

<p style="text-align:justify;">HIP streams enable asynchronous execution and overlap of operations, which is essential for achieving high performance in GPU applications. A stream is an ordered sequence of operations that execute on the GPU - operations within a stream execute in the order they are enqueued, but different streams can execute concurrently. The hipStreamCreate call creates a new stream object that will manage a queue of GPU operations. The hipMemcpyAsync calls then enqueue memory transfer operations from host to device within this stream - these transfers copy the input data from host arrays hA and hB to the corresponding device arrays dA and dB. The "Async" suffix indicates that these transfers are non-blocking: the CPU immediately returns from the hipMemcpyAsync call and continues executing subsequent code while the GPU hardware handles the actual data transfer in the background. This asynchronous behavior is crucial because it allows the CPU to prepare additional work (like launching kernels) while transfers are still in progress, enabling overlap between data movement and computation. All operations in the same stream execute sequentially in the order enqueued, ensuring that the kernel won't execute until these memory transfers complete, maintaining program correctness while maximizing hardware utilization.</p>

## 2.6 Events for Kernel Timing

```cpp
hipEvent_t start, stop;
HIP_CHECK( hipEventCreate(&start) );
HIP_CHECK( hipEventCreate(&stop) );

HIP_CHECK( hipEventRecord(start, stream) );

dim3 block(256);
dim3 grid((N + block.x - 1) / block.x);

hipLaunchKernelGGL(
    vectorAdd, grid, block, 0, stream,
    dA, dB, dC, N
);

HIP_CHECK( hipEventRecord(stop, stream) );
HIP_CHECK( hipEventSynchronize(stop) );

float elapsedMs = 0.0f;
HIP_CHECK( hipEventElapsedTime(&elapsedMs, start, stop) );
std::cout << "Kernel time (ms): " << elapsedMs << "\n";
```

<p style="text-align:justify;">This section demonstrates GPU kernel timing using HIP events and shows how to launch GPU kernels with proper execution configuration. Events are lightweight synchronization markers that can be inserted into a stream to record timestamps on the GPU timeline. The program creates two events (start and stop) using hipEventCreate, then records the start event into the stream just before kernel execution. The kernel launch configuration is specified using dim3 structures: block(256) creates blocks of 256 threads each, and the grid size is calculated to ensure enough threads cover all N elements - the formula `(N + block.x - 1) / block.x` performs ceiling division to handle cases where N is not evenly divisible by the block size. The hipLaunchKernelGGL macro launches the vectorAdd kernel with this configuration, passing the device pointers and problem size as kernel arguments. The kernel executes asynchronously on the GPU while the CPU continues. Immediately after launching the kernel, the stop event is recorded in the stream. The hipEventSynchronize call blocks the CPU until the stop event completes, ensuring the kernel has finished executing. Finally, hipEventElapsedTime calculates the precise time interval between the start and stop events as measured on the GPU, providing accurate kernel execution time in milliseconds. This timing mechanism is more accurate than CPU-side timing because it measures only GPU execution time, excluding any host overhead or data transfer time.</p>

## 2.7 Result Retrieval and Validation

```cpp
HIP_CHECK( hipMemcpyAsync(hC, dC, size,
                          hipMemcpyDeviceToHost, stream) );
HIP_CHECK( hipStreamSynchronize(stream) );

bool pass = true;
for (int i = 0; i < N; i++) {
    if (hC[i] != hA[i] + hB[i]) {
        pass = false;
        break;
    }
}

std::cout << "Vector Add Result: "
          << (pass ? "PASS" : "FAIL") << "\n";
```

<p style="text-align:justify;">After the kernel completes execution on the GPU, the results must be transferred back to host memory for validation and further use. The hipMemcpyAsync call copies data from the device output array dC back to the host array hC, transferring in the device-to-host direction. This transfer is also asynchronous and enqueued into the same stream, maintaining the proper ordering of operations. The critical hipStreamSynchronize call then blocks the CPU thread until all operations in the stream have completed - this includes the memory transfers and kernel execution. This synchronization is absolutely essential because without it, the CPU might attempt to read the results from hC before the GPU has finished writing them, leading to undefined behavior or incorrect results. Once synchronization completes, the program validates correctness by iterating through all elements and comparing each GPU-computed result hC[i] with the expected value hA[i] + hB[i]. If any element doesn't match, the validation fails. This verification step confirms that the entire GPU pipeline worked correctly: data was transferred to the device, the kernel executed properly with correct results, and data was successfully retrieved back to the host. The PASS/FAIL output provides immediate feedback about the correctness of the GPU computation.</p>

## 2.8 Resource Cleanup

```cpp
HIP_CHECK( hipEventDestroy(start) );
HIP_CHECK( hipEventDestroy(stop) );
HIP_CHECK( hipStreamDestroy(stream) );

HIP_CHECK( hipFree(dA) );
HIP_CHECK( hipFree(dB) );
HIP_CHECK( hipFree(dC) );

delete[] hA;
delete[] hB;
delete[] hC;

HIP_CHECK( hipDeviceReset() );
```

<p style="text-align:justify;">Proper resource cleanup is essential in GPU programming to prevent memory leaks and ensure system stability. This cleanup section systematically releases all resources in reverse order of their allocation. First, the HIP events are destroyed using hipEventDestroy, freeing any internal resources associated with timing infrastructure. Next, the stream is destroyed with hipStreamDestroy, releasing the command queue and associated metadata. The device memory allocations are then freed using hipFree for each of the three GPU arrays - this returns the GPU memory to the available pool for future allocations. The host arrays are deleted using standard C++ delete[] operators, freeing the system RAM. Finally, hipDeviceReset performs a complete reset of the GPU device context, which releases any remaining resources, resets all GPU state, and ensures a clean exit from the HIP runtime. This reset is particularly important in development environments where you might run multiple GPU programs sequentially - it ensures each program starts with a fresh GPU state. While modern operating systems will eventually reclaim GPU resources when a process terminates, explicit cleanup demonstrates good programming practice, makes resource management predictable, and can prevent issues in long-running applications or when running multiple GPU operations within the same process.</p>

---

# Step 3: Building and Running the HIP Application

Build the complete HIP application and execute it to validate correct end-to-end GPU execution.

## 3.1 Compile for your specific GPU architecture

This compiles the HIP source file and generates an executable named `lab`. 

**IMPORTANT:** Replace `gfx942` below with **your actual GPU architecture** detected in Step 1.3.

**Compiler flags:**
- `--offload-arch=<your_arch>` - Target your GPU architecture
  - Use the architecture you detected earlier (e.g., gfx1100, gfx1151, gfx942, gfx90a)
  - **Example for Radeon RX 7000 series:** `--offload-arch=gfx1100`
  - **Example for Ryzen iGPU:** `--offload-arch=gfx1151`
- `lab.cpp` - Source file
- `-o lab` - Output executable name

**Why this matters:** The compiled binary contains GPU instructions specific to your architecture. A mismatch will cause FAIL or crashes.

**Note:** This compilation process works identically across all AMD GPU architectures.

In [ ]:
!hipcc --offload-arch=gfx942 lab.cpp -o lab

## 3.2 Execute the HIP program

This executes the compiled HIP application on the GPU.

**Expected output:**
- GPU hardware information (name, architecture, compute units, memory)
- Kernel execution time in milliseconds
- Validation result: PASS or FAIL

In [ ]:
!./lab

## 3.3 Understanding the Output

The execution output confirms successful end-to-end GPU execution:

**Output components:**

1. **GPU Hardware Information:**
   - `GPU: <name>` - Your GPU device name
   - `Architecture: <arch>` - GPU architecture identifier (e.g., gfx1151, gfx1100, gfx942)
     - The architecture string may include feature flags like `sramecc+`, `xnack-`
   - `Compute Units: <count>` - Number of parallel compute units available
   - `Global Memory (MB): <size>` - Total GPU memory available

2. **Kernel Execution Time:**
   - `Kernel time (ms): <value>` - GPU execution time measured with HIP events
   - Value varies based on GPU performance and workload
   - Confirms successful kernel launch and completion

3. **Validation Result:**
   - `Vector Add Result: PASS` - GPU computation is correct ✅
   - `Vector Add Result: FAIL` - Indicates a problem (usually architecture mismatch) ❌
   - **If you see FAIL:** Verify that `--offload-arch` matches your detected GPU architecture

**What this confirms:**
- HIP runtime initialized successfully
- GPU device detected and selected
- Memory allocated and transferred correctly
- Kernel executed on GPU with correct architecture
- Results validated on host
- Complete host-driven execution flow works correctly

**Note:** The exact output values (compute units, memory size, execution time) will vary depending on your specific AMD GPU model, but the workflow is identical across all supported architectures.

---

# Summary

In this lab, you:

✅ Verified the HIP development environment

✅ Reviewed a complete HIP application demonstrating:
   - GPU kernel definition and thread indexing
   - Host-driven execution model
   - Explicit memory management
   - Asynchronous execution with streams
   - Event-based timing
   - Synchronization and validation

✅ Built and executed the program on your AMD GPU

✅ Observed GPU hardware characteristics and verified correct execution

## Key Takeaways

You now understand how HIP runtime APIs orchestrate GPU computation through:
- Runtime initialization and device selection
- Explicit memory allocation and transfers
- Asynchronous kernel launches
- Synchronization to ensure correctness
- Proper resource cleanup
- **Architecture-specific compilation** - matching `--offload-arch` to your actual GPU

This foundation prepares you for more advanced HIP programming concepts and performance optimization techniques.

**Note:** This workflow applies uniformly across all AMD GPU families - from consumer graphics cards to professional compute accelerators - making HIP a portable solution for GPU computing.